In [1]:
# -*- coding: utf-8 -*-
"""
Created on Wed Oct 25 18:16:33 2023

@author: sowmya
"""

import cv2
import numpy as np
from keras.models import load_model
import tensorflow as tf
import keras.backend as K


def calc_cm_vals(y_true, y_pred, class_index=None):
    pred = tf.argmax(y_pred, axis=1)
    true = tf.reshape(y_true, (-1,))

    tp = tf.reduce_sum(tf.cast(tf.logical_and(tf.equal(true, class_index),
                                              tf.equal(pred, class_index)),
                                               tf.float32))
    fp = tf.reduce_sum(tf.cast(tf.logical_and(tf.not_equal(true, class_index),
                                              tf.equal(pred, class_index)),
                                               tf.float32))
    fn = tf.reduce_sum(tf.cast(tf.logical_and(tf.equal(true, class_index),
                                             tf.not_equal(pred, class_index)),
                                              tf.float32))
    return tp, fp, fn

def recall(y_true, y_pred, class_index=0):
    tp, fp, fn = calc_cm_vals(y_true, y_pred, class_index)
    return tp / (tp + fn + K.epsilon())

def precision(y_true, y_pred, class_index=1):
    tp, fp, fn = calc_cm_vals(y_true, y_pred, class_index)
    return tp / (tp + fp + K.epsilon())






def preprocess_img(image, target_size):
    if image is not None:
        height, width, channels = image.shape
        if width<height :
            image = cv2.rotate(image, cv2.ROTATE_90_CLOCKWISE)
    preprocessed_image = cv2.resize(image, target_size)
    converted_image = cv2.cvtColor(preprocessed_image, cv2.COLOR_BGR2RGB)
    reshaped_image = converted_image.reshape(1, *converted_image.shape)
    image = reshaped_image.astype(np.float32)
    image = image / 255.0
    return image

def predict_image(preprocessed_img, model, label_map):
    prediction = model.predict(preprocessed_img)
    prediction = np.array([np.argmax(pred) for pred in prediction])
    reverse_label_map = {idx: img_type for img_type, idx in label_map.items()}
    predicted_class = reverse_label_map[prediction[0]]
    return predicted_class





def main(img_path = ''):
    model_path = r"/content/drive/MyDrive/Sowmya /models/Sardine/v3/densenet201_model.h5"
    label_map = {'Bad': 0, 'Good': 1}
    target_size = (280, 180)

    custom_metrics = {'recall_0':recall, 'precision_1':precision}
    model = load_model(model_path, custom_objects=custom_metrics)

    img = cv2.imread(img_path)
    preprocessed_img = preprocess_img(img, target_size)
    predicted_class = predict_image(preprocessed_img, model, label_map)
    print(predicted_class)


In [2]:
# Input image must be segmented with black background

# img_path = 'path to segmented image'
img_path = "/content/drive/MyDrive/Sowmya /models/Sardine/20230528093954652_sardine_good_(0).png"
main(img_path)

1/1 [==============================] - 7s 7s/step
Good
